# Samaritan solver on an A100 — Qwen3.8-27B (turnkey)

Serves **Qwen3.8-27B** (Q8_0 GGUF) on this Colab A100 via **Ollama**, and
exposes it as an OpenAI-compatible endpoint your **local** Samaritan harness
points at — no code change, just `SAMARITAN_URL`.

**Why Ollama, not vLLM:** vLLM can't run FP8 MoE on the A100 (Ampere has no
FP8 tensor cores), and this model's newer arch isn't confirmed on the vLLM
build here. The GGUF + Ollama path is proven (11.5M downloads) and bundles
its own CUDA, so there's no build fight with Colab's CUDA-13. Q8_0 (~30 GB)
fits the 40 GB card with full GPU offload.

**Before running:** Runtime → Change runtime type → **A100 GPU** (Pro+).
Then Runtime → **Run all**. The tunnel cell prints the line to paste locally.

**Honest limits:** Colab is not 24/7 (Pro+ background ~24 h, can drop) — good
for eval pushes and self-training, not a deployment. **Ollama has no API-key**
auth, so the public tunnel URL is the *only* guard — don't share it, and run
the shutdown cell (or stop the runtime) when done; a forgotten A100 burns
compute units.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

## 2. Install Ollama and start the server

Ollama ships its own CUDA runtime, so nothing compiles against Colab's stack.
Colab has no systemd, so we start the daemon ourselves and tell it to keep the
model resident (`OLLAMA_KEEP_ALIVE=-1`) so eval runs don't trigger reloads.

In [ ]:
import os, subprocess, time, urllib.request
!curl -fsSL https://ollama.com/install.sh | sh
env = {**os.environ, 'OLLAMA_HOST': '127.0.0.1:11434', 'OLLAMA_KEEP_ALIVE': '-1'}
srv = subprocess.Popen(['ollama', 'serve'], stdout=open('ollama.log', 'w'),
                       stderr=subprocess.STDOUT, env=env)
up = False
for _ in range(30):
    try:
        if urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=3).status == 200:
            up = True; break
    except Exception:
        time.sleep(2)
print('ollama daemon:', 'UP' if up else 'not up yet — see ollama.log')

## 3. Pull Qwen3.8-27B (Q8_0) and alias it to `samaritan-playout`

`qwen3.8:27b-q8_0` is ~30 GB and fits the 40 GB A100 with room; there is no
Q6_K in the Ollama library for the 27B, and bf16 (56 GB) won't fit — so Q8_0
is the high-fidelity tag that runs here. We copy it to `samaritan-playout`
(the name the harness asks for) and bake a 16k context into it via a Modelfile,
because this model **thinks hard by default** and we don't want long traces
truncated. 16k is safe on 40 GB — raise it if you have headroom, lower it if
you OOM. The pull is ~30 GB; Colab's pipe is fast, but give it a few minutes.

In [ ]:
!ollama pull qwen3.8:27b-q8_0
with open('Modelfile', 'w') as f:
    f.write('FROM qwen3.8:27b-q8_0\nPARAMETER num_ctx 16384\n')
!ollama create samaritan-playout -f Modelfile
!ollama list

## 4. Expose it with a cloudflared tunnel

Publishes Ollama's port 11434 as a public `https://…trycloudflare.com` URL.
**Reminder:** Ollama has no auth, so this URL is the only thing protecting the
endpoint — treat it like a secret and stop the runtime when you're done.

In [ ]:
import subprocess, re, time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
cf = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
                      stdout=open('cf.log', 'w'), stderr=subprocess.STDOUT)
public = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('cf.log').read())
    if m: public = m.group(0); break
if not public:
    print('no tunnel URL yet — tail of cf.log:'); print(open('cf.log').read()[-1500:])
else:
    print('Tunnel up. Paste into your LOCAL terminal:\n')
    print(f'  $env:SAMARITAN_URL = "{public}/v1"        # PowerShell')
    print(f'  $env:SAMARITAN_API_KEY = "ollama"           # any value; Ollama ignores it')
    print(f'\n  export SAMARITAN_URL="{public}/v1"          # bash')
    print(f'  export SAMARITAN_API_KEY="ollama"')

## 5. Self-test (also warms the model into VRAM)

The first call loads ~30 GB onto the GPU, so allow a minute or two; after that
it stays resident. The reply may include a long `<think>` block — that's the
model reasoning, and the harness strips it.

In [ ]:
import urllib.request, json
body = json.dumps({'model': 'samaritan-playout',
    'messages': [{'role': 'user', 'content': 'What is 6 times 7? Reply with just the number.'}],
    'max_tokens': 1024, 'temperature': 1.0, 'top_p': 0.95}).encode()
req = urllib.request.Request(f'{public}/v1/chat/completions', data=body,
    headers={'Content-Type': 'application/json'})
r = json.loads(urllib.request.urlopen(req, timeout=300).read())
print(r['choices'][0]['message']['content'][-600:])

## Now, on your laptop

```powershell
$env:SAMARITAN_URL = "https://<random>.trycloudflare.com/v1"   # from cell 4
$env:SAMARITAN_API_KEY = "ollama"                              # any value
$env:MAX_TOKENS = "8192"      # Qwen3.8 thinks a lot — give the trace room
$env:DATASET = "$env:USERPROFILE\models\reasoning\gsm-symbolic-p2.jsonl"
cargo run -p samaritan-run --example reason_eval
```

**Sampling note:** Qwen recommends **temp 1.0, top_p 0.95, top_k 20, repeat 1.0**
for this model in thinking mode. `reason_eval` currently sends temp 0.6 / repeat
1.1 (tuned for the local 4B) — fine for a first read, but ask me to make the
example's sampling env-configurable and you can run this model at its own
recommended settings.

Leave this notebook running. This is the first real read from a *capable*
substrate — it should clear the local 4B's 2/10 on GSM-Symbolic p2.

## Optional: self-training fine-tune here too

Independent of the served 27B: generate the verified set locally
(`selftrain_export`), upload it here, and QDoRA-train the **local 4B** on this
A100. Uploads `reasoning-selftrain.jsonl` + `qdora_deviant.py` + `requirements.txt`
from your machine (or clone the repo).

In [ ]:
# from google.colab import files; files.upload()   # reasoning-selftrain.jsonl + qdora_deviant.py + requirements.txt
# !pip -q install -r requirements.txt
# !python qdora_deviant.py reasoning-selftrain.jsonl --base-model Qwen/Qwen3-4B-Thinking-2507 --allow-small
# then download the adapter and convert to GGUF (see training/README.md)
print('uncomment the lines above to train; see training/README.md')

## Shutdown (run when done)

In [ ]:
for name in ['cf', 'srv']:
    try: globals()[name].terminate()
    except Exception: pass
print('stopped the tunnel and Ollama. Also: Runtime -> Disconnect and delete runtime.')